In [1]:
import os
import gc
import mne
import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
import collections
import time
from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTETomek
from scipy import signal
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset, random_split, WeightedRandomSampler
import torchvision.transforms as transforms
import timm
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score, roc_curve, auc
from sklearn.model_selection import StratifiedShuffleSplit
from io import BytesIO

# 1. EEG PreProcessing & Spectogram Generation

In [3]:
def apply_data_augmentation(window_data):
    """Apply signal-level augmentations to EEG data."""
    augmented_windows = []
    # Original window
    augmented_windows.append(window_data)
    # Time shifting
    shift = np.random.randint(-100, 100)
    shifted = np.roll(window_data, shift, axis=1)
    augmented_windows.append(shifted)
    # Amplitude scaling
    scale = np.random.uniform(0.8, 1.2)
    scaled = window_data * scale
    augmented_windows.append(scaled)
    # Add Gaussian noise
    noise = np.random.normal(0, 0.01, window_data.shape)
    noisy = window_data + noise
    augmented_windows.append(noisy)
    # Frequency shifting using phase shift
    freq_shift = np.random.uniform(-2, 2)
    t = np.arange(window_data.shape[1])
    phase_shift = np.exp(2j * np.pi * freq_shift * t / len(t))
    freq_shifted = window_data * phase_shift
    augmented_windows.append(np.real(freq_shifted))
    return augmented_windows

In [4]:
def generate_synthetic_seizures(window_data, num_synthetic=2):
    """Generate synthetic seizure windows using interpolation and mixing."""
    synthetic_windows = []
    for _ in range(num_synthetic):
        warp_factor = np.random.uniform(0.9, 1.1)
        new_length = int(window_data.shape[1] * warp_factor)
        warped = signal.resample(window_data, new_length)
        if warped.shape[1] != window_data.shape[1]:
            warped = signal.resample(warped, window_data.shape[1])
        amplitude_mod = np.random.uniform(0.85, 1.15)
        noise_level = np.random.uniform(0.01, 0.03)
        synthetic = warped * amplitude_mod + np.random.normal(0, noise_level, warped.shape)
        synthetic_windows.append(synthetic)
    return synthetic_windows

In [5]:
# Seizure data for CHB05 (only files with seizures)
seizure_data = [
    ["chb05_06.edf", 417, 532],
    ["chb05_13.edf", 1086, 1196],
    ["chb05_16.edf", 2317, 2413],
    ["chb05_17.edf", 2451, 2571],
    ["chb05_22.edf", 2348, 2465]
]

# Set EEG directory to CHB05 folder
eeg_directory = "/kaggle/input/seizure-epilepcy-chb-mit-eeg-dataset-pediatric/chb-mit-scalp-eeg-database-1.0.0/chb05"
output_dir = "spectrogram_data"
os.makedirs(output_dir, exist_ok=True)

csv_data = []
spectrogram_counter = 0

print("Starting enhanced spectrogram generation for CHB05...")
for file_name in sorted(os.listdir(eeg_directory)):
    if not file_name.endswith(".edf"):
        continue
    file_path = os.path.join(eeg_directory, file_name)
    print(f"Processing file: {file_name}")
    raw = mne.io.read_raw_edf(file_path, preload=True, verbose=False)
    montage = mne.channels.make_standard_montage('standard_1020')
    raw.set_montage(montage, on_missing='ignore')
    raw.set_eeg_reference(ref_channels='average')
    raw.filter(l_freq=1.0, h_freq=None, verbose=False)
    raw.notch_filter(freqs=[50], verbose=False)
    
    sfreq = raw.info['sfreq']
    total_samples = raw.n_times
    
    seizure_info = next((s for s in seizure_data if s[0] == file_name), None)
    # Process files without seizure info as non-seizure windows
    if seizure_info is None:
        for start_sample in range(0, total_samples, int(30*sfreq)):
            end_sample = start_sample + int(30*sfreq)
            if end_sample > total_samples:
                break
            window_data = raw.get_data(start=start_sample, stop=end_sample)
            window_data_flat = window_data.flatten()
            stft_result = librosa.stft(window_data_flat, n_fft=128, hop_length=256)
            spectrogram = np.abs(stft_result)
            spectrogram_db = librosa.amplitude_to_db(spectrogram, ref=np.max)
            buf = BytesIO()
            fig, ax = plt.subplots(figsize=(4, 4), dpi=100)
            ax.imshow(spectrogram_db, aspect='auto', origin='lower', cmap='viridis')
            ax.axis('off')
            plt.savefig(buf, format='png', bbox_inches='tight', pad_inches=0)
            plt.close(fig)
            buf.seek(0)
            img = Image.open(buf).convert("RGB")
            solarized = ImageOps.solarize(img, threshold=128)
            spectrogram_filename = os.path.join(output_dir, f"spectrogram_{spectrogram_counter}.png")
            solarized.save(spectrogram_filename)
            csv_data.append([spectrogram_filename, 0])
            spectrogram_counter += 1
        gc.collect()
        continue

    seizure_start, seizure_end = seizure_info[1], seizure_info[2]
    sph_start_sample = max(int((seizure_start - 600) * sfreq), 0)
    sop_start_sample = int(seizure_start * sfreq)
    sop_end_sample = min(int((seizure_start + 1800) * sfreq), total_samples)
    window_size = int(30 * sfreq)
    
    for start_sample in range(sph_start_sample, sop_end_sample, window_size):
        end_sample = start_sample + window_size
        if end_sample > total_samples:
            continue
        start_time = start_sample / sfreq
        
        if seizure_start <= start_time < seizure_end:
            label = 1
            window_data = raw.get_data(start=start_sample, stop=end_sample)
            augmented_windows = apply_data_augmentation(window_data)
            synthetic_windows = generate_synthetic_seizures(window_data)
            all_windows = augmented_windows + synthetic_windows
            for aug_window in all_windows:
                window_data_flat = aug_window.flatten()
                stft_result = librosa.stft(window_data_flat, n_fft=128, hop_length=256)
                spectrogram = np.abs(stft_result)
                spectrogram_db = librosa.amplitude_to_db(spectrogram, ref=np.max)
                mask_param = int(spectrogram_db.shape[0] * 0.1)
                freq_mask_idx = np.random.randint(0, spectrogram_db.shape[0] - mask_param)
                spectrogram_db[freq_mask_idx:freq_mask_idx + mask_param, :] = spectrogram_db.min()
                buf = BytesIO()
                fig, ax = plt.subplots(figsize=(4, 4), dpi=100)
                ax.imshow(spectrogram_db, aspect='auto', origin='lower', cmap='viridis')
                ax.axis('off')
                plt.savefig(buf, format='png', bbox_inches='tight', pad_inches=0)
                plt.close(fig)
                buf.seek(0)
                img = Image.open(buf).convert("RGB")
                solarized = ImageOps.solarize(img, threshold=128)
                spectrogram_filename = os.path.join(output_dir, f"spectrogram_{spectrogram_counter}.png")
                solarized.save(spectrogram_filename)
                csv_data.append([spectrogram_filename, label])
                spectrogram_counter += 1
        else:
            label = 0
            window_data = raw.get_data(start=start_sample, stop=end_sample)
            window_data_flat = window_data.flatten()
            stft_result = librosa.stft(window_data_flat, n_fft=128, hop_length=256)
            spectrogram = np.abs(stft_result)
            spectrogram_db = librosa.amplitude_to_db(spectrogram, ref=np.max)
            buf = BytesIO()
            fig, ax = plt.subplots(figsize=(4, 4), dpi=100)
            ax.imshow(spectrogram_db, aspect='auto', origin='lower', cmap='viridis')
            ax.axis('off')
            plt.savefig(buf, format='png', bbox_inches='tight', pad_inches=0)
            plt.close(fig)
            buf.seek(0)
            img = Image.open(buf).convert("RGB")
            solarized = ImageOps.solarize(img, threshold=128)
            spectrogram_filename = os.path.join(output_dir, f"spectrogram_{spectrogram_counter}.png")
            solarized.save(spectrogram_filename)
            csv_data.append([spectrogram_filename, label])
            spectrogram_counter += 1


        gc.collect()

print("Enhanced spectrogram generation complete!")
csv_filename = "spectrogram_labels.csv"
df = pd.DataFrame(csv_data, columns=["File_Path", "Label"])
df.to_csv(csv_filename, index=False)
print(f"CSV saved: {csv_filename}")


Starting enhanced spectrogram generation for CHB05...
Processing file: chb05_01.edf


<ipython-input-5-e293589943ea>:24: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(file_path, preload=True, verbose=False)


EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.
Processing file: chb05_02.edf


<ipython-input-5-e293589943ea>:24: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(file_path, preload=True, verbose=False)


EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.
Processing file: chb05_03.edf


...

EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.
Processing file: chb05_38.edf


<ipython-input-5-e293589943ea>:24: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(file_path, preload=True, verbose=False)


EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.
Processing file: chb05_39.edf


<ipython-input-5-e293589943ea>:24: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(file_path, preload=True, verbose=False)


EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.
Enhanced spectrogram generation complete!
CSV saved: spectrogram_labels.csv


In [6]:
import pandas as pd

# Define the CSV file path
csv_path = "/kaggle/working/spectrogram_labels.csv"  # Ensure this is the correct file path

# Load the CSV file
df = pd.read_csv(csv_path)

# Check the actual column names
print("Column names:", df.columns)

# Ensure you are modifying the correct column
if "File_Path" in df.columns:
    df["File_Path"] = df["File_Path"].str.replace("spectrogram_data/", "", regex=False)
else:
    print("Error: 'File_Path' column not found!")

# Save the updated CSV file
df.to_csv("/kaggle/working/labels.csv", index=False)

print("File paths updated successfully!")

Column names: Index(['File_Path', 'Label'], dtype='object')
File paths updated successfully!


# 2. Enhanced Dataset with Advanced Augmentation

In [7]:
class AdvancedSpectrogramDataset(Dataset):
    def __init__(self, csv_file, transform=None, augment_positive=True):
        self.data_df = pd.read_csv(csv_file)
        self.transform = transform
        self.augment_positive = augment_positive
        self.positive_transform = transforms.Compose([
            transforms.RandomApply([
                transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
            ], p=0.7),
            transforms.RandomApply([
                transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0))
            ], p=0.5),
            transforms.RandomAffine(degrees=10, translate=(0.1, 0.1), scale=(0.9, 1.1)),
        ])
    
    def __len__(self):
        return len(self.data_df)
    
    def __getitem__(self, idx):
        img_path = self.data_df.iloc[idx, 0]
        label = int(self.data_df.iloc[idx, 1])
        image = Image.open(img_path).convert("RGB")
        if label == 1 and self.augment_positive:
            image = self.positive_transform(image)
        if self.transform:
            image = self.transform(image)
        return image, label
base_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

csv_filename = "/kaggle/working/labels.csv"
dataset = AdvancedSpectrogramDataset(csv_filename, transform=base_transform)

# --- Three-way stratified split: Train (60%), Val (20%), Test (20%) ---
labels = [dataset[i][1] for i in range(len(dataset))]
sss1 = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
for train_val_idx, test_idx in sss1.split(np.zeros(len(labels)), labels):
    train_val_indices = train_val_idx
    test_indices = test_idx

train_val_labels = [labels[i] for i in train_val_indices]
sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
for train_idx, val_idx in sss2.split(np.zeros(len(train_val_labels)), train_val_labels):
    train_indices = [train_val_indices[i] for i in train_idx]
    val_indices = [train_val_indices[i] for i in val_idx]

train_dataset = Subset(dataset, train_indices)
val_dataset = Subset(dataset, val_indices)
test_dataset = Subset(dataset, test_indices)

train_labels = [dataset[i][1] for i in train_indices]
counter = collections.Counter(train_labels)
num_classes = 2
total_count = len(train_labels)
class_weights = {cls: total_count / (num_classes * counter[cls]) for cls in counter}
sample_weights = [class_weights[dataset[i][1]] for i in train_indices]
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

train_loader = DataLoader(train_dataset, batch_size=8, sampler=sampler, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False, num_workers=2)

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/spectrogram_0.png'

# 3. Model Architecture

In [47]:
class EnhancedCombinedModel(nn.Module):
    def __init__(self, num_classes=2):
        super(EnhancedCombinedModel, self).__init__()
        self.deit = timm.create_model('deit3_small_patch16_224', pretrained=True)
        self.deit.reset_classifier(0)
        # Use EfficientNet B3 instead of B0
        self.effnet = timm.create_model('efficientnet_b3', pretrained=True)
        if hasattr(self.effnet, 'classifier'):
            self.effnet.classifier = nn.Identity()
        else:
            self.effnet.fc = nn.Identity()
        sample_input = torch.randn(1, 3, 224, 224)
        with torch.no_grad():
            feat_deit = self.deit.forward_features(sample_input)
            feat_eff = self.effnet.forward_features(sample_input) if hasattr(self.effnet, 'forward_features') else self.effnet(sample_input)
        if feat_deit.dim() == 3:
            deit_feat_dim = feat_deit.shape[-1]
        else:
            deit_feat_dim = feat_deit.shape[1]
        if feat_eff.dim() == 3:
            effnet_feat_dim = feat_eff.shape[-1]
        else:
            effnet_feat_dim = feat_eff.shape[1]
        self.fusion = nn.Sequential(
            nn.Linear(deit_feat_dim + effnet_feat_dim, 1024),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3)
        )
        self.classifier = nn.Linear(256, num_classes)
    
    def forward(self, x):
        x = nn.functional.interpolate(x, size=(224, 224), mode='bilinear', align_corners=False)
        feat_deit = self.deit.forward_features(x)
        if feat_deit.dim() == 3:
            feat_deit = feat_deit[:, 0, :]
        elif feat_deit.dim() == 4:
            feat_deit = nn.functional.adaptive_avg_pool2d(feat_deit, 1).view(x.size(0), -1)
        feat_eff = self.effnet.forward_features(x) if hasattr(self.effnet, 'forward_features') else self.effnet(x)
        if feat_eff.dim() == 3:
            feat_eff = feat_eff[:, 0, :]
        elif feat_eff.dim() == 4:
            feat_eff = nn.functional.adaptive_avg_pool2d(feat_eff, 1).view(x.size(0), -1)
        fused = torch.cat([feat_deit, feat_eff], dim=1)
        fused = self.fusion(fused)
        out = self.classifier(fused)
        return out

# 4. Training Components

In [48]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction
        
    def forward(self, inputs, targets):
        ce_loss = nn.functional.cross_entropy(inputs, targets, reduction='none', weight=self.alpha)
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        return focal_loss

def mixup_data(x, y, alpha=0.4, device='cuda'):
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1
    batch_size = x.size()[0]
    index = torch.randperm(batch_size).to(device)
    mixed_x = lam * x + (1 - lam) * x[index, :]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

class OnlineSMOTEBatch:
    def __init__(self, k_neighbors=5):
        self.k_neighbors = k_neighbors
    
    def generate_synthetic_samples(self, features, labels, num_synthetic):
        positive_mask = (labels == 1)
        if positive_mask.sum().item() < 2:
            return features, labels
        positive_samples = features[positive_mask]
        synthetic_samples = []
        for _ in range(num_synthetic):
            idx = np.random.randint(0, positive_samples.size(0))
            nn_idx = np.random.randint(0, positive_samples.size(0))
            point = positive_samples[idx]
            nn = positive_samples[nn_idx]
            alpha_val = np.random.random()
            synthetic_sample = point + alpha_val * (nn - point)
            synthetic_samples.append(synthetic_sample)
        if synthetic_samples:
            synthetic_tensor = torch.stack(synthetic_samples)
            synthetic_labels = torch.ones(synthetic_tensor.size(0), dtype=torch.long, device=features.device)
            features = torch.cat([features, synthetic_tensor], dim=0)
            labels = torch.cat([labels, synthetic_labels], dim=0)
        return features, labels

# 5. Training Loop

In [50]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = EnhancedCombinedModel(num_classes=2)
model.to(device)

online_smote = OnlineSMOTEBatch(k_neighbors=5)

alpha_tensor = torch.tensor([class_weights[i] for i in range(num_classes)], dtype=torch.float32).to(device)
criterion = FocalLoss(alpha=alpha_tensor, gamma=2.0, reduction='mean')

optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)
scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=1e-3,
    epochs=50,
    steps_per_epoch=len(train_loader),
    pct_start=0.3,
    anneal_strategy='cos'
)

best_val_loss = float('inf')
patience = 7
patience_counter = 0
best_model_path = 'best_seizure_model.pth'

def train_epoch(model, train_loader, criterion, optimizer, scheduler, device, mixup_alpha=0.4):
    model.train()
    running_loss = 0.0
    all_preds = []
    all_labels = []
    for inputs, labels in train_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)
        if (labels == 1).any():
            inputs, labels = online_smote.generate_synthetic_samples(inputs, labels, num_synthetic=2)
        inputs, targets_a, targets_b, lam = mixup_data(inputs, labels, alpha=mixup_alpha, device=device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = mixup_criterion(criterion, outputs, targets_a, targets_b, lam)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        running_loss += loss.item() * inputs.size(0)
        preds = torch.argmax(outputs, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    epoch_loss = running_loss / len(train_loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    return epoch_loss, epoch_acc

def validate(model, val_loader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * inputs.size(0)
            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    val_loss = running_loss / len(val_loader.dataset)
    val_acc = accuracy_score(all_labels, all_preds)
    cm = confusion_matrix(all_labels, all_preds)
    return val_loss, val_acc, cm

print("Starting enhanced training loop...")
num_epochs = 21
for epoch in range(num_epochs):
    start_time = time.time()
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, scheduler, device, mixup_alpha=0.4)
    val_loss, val_acc, cm = validate(model, val_loader, criterion, device)
    elapsed = time.time() - start_time
    print(f"\nEpoch {epoch+1}/{num_epochs} (Time: {elapsed:.1f}s):")
    print(f"  Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"  Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4f}")
    print("  Confusion Matrix (Validation):")
    print(cm)
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save(model.state_dict(), best_model_path)
        print("  Model improved - saving checkpoint")
    """else:
        patience_counter += 1
        if patience_counter >= patience:
            print("  Early stopping triggered")
            break"""
    torch.cuda.empty_cache()
    gc.collect()

torch.save(model.state_dict(), "DeiT3EfficientNet_model_weights.pth")
print("Model weights saved!")

Starting enhanced training loop...

Epoch 1/21 (Time: 61.1s):
  Train Loss: 0.6273 | Train Acc: 0.6549
  Val Loss:   0.1544 | Val Acc:   0.8313
  Confusion Matrix (Validation):
[[726 153]
 [  0  28]]
  Model improved - saving checkpoint

Epoch 2/21 (Time: 61.2s):
  Train Loss: 0.2052 | Train Acc: 0.7483
  Val Loss:   0.0180 | Val Acc:   0.9614
  Confusion Matrix (Validation):
[[844  35]
 [  0  28]]
  Model improved - saving checkpoint

Epoch 3/21 (Time: 61.0s):
  Train Loss: 0.2586 | Train Acc: 0.6690
  Val Loss:   0.0294 | Val Acc:   0.9989
  Confusion Matrix (Validation):
[[878   1]
 [  0  28]]

Epoch 4/21 (Time: 61.6s):
  Train Loss: 0.2342 | Train Acc: 0.7193
  Val Loss:   0.0319 | Val Acc:   0.4002
  Confusion Matrix (Validation):
[[335 544]
 [  0  28]]

Epoch 5/21 (Time: 61.3s):
  Train Loss: 0.2612 | Train Acc: 0.7163
  Val Loss:   0.0107 | Val Acc:   0.8986
  Confusion Matrix (Validation):
[[787  92]
 [  0  28]]
  Model improved - saving checkpoint

Epoch 6/21 (Time: 61.1s):
  

# 6. Final Evaluation on Test Set & Results.

In [ ]:
model.load_state_dict(torch.load(best_model_path))
model.eval()
y_true, y_pred, y_scores = [], [], []
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)
        outputs = model(inputs)
        probabilities = torch.softmax(outputs, dim=1)
        _, predictions = torch.max(outputs, 1)
        y_true.extend(labels.cpu().numpy())
        y_pred.extend(predictions.cpu().numpy())
        y_scores.extend(probabilities[:, 1].cpu().numpy())

y_true = np.array(y_true)
y_pred = np.array(y_pred)
y_scores = np.array(y_scores)

final_accuracy = accuracy_score(y_true, y_pred)
final_precision = precision_score(y_true, y_pred, average='weighted', zero_division=0)
final_recall = recall_score(y_true, y_pred, average='weighted', zero_division=0)
final_f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
final_cm = confusion_matrix(y_true, y_pred)

if final_cm.shape == (2, 2):
    tn, fp, fn, tp = final_cm.ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    total_non_seizure_windows = tn + fp
    total_non_seizure_hours = (total_non_seizure_windows * 30) / 3600
    fpr_per_hour = fp / total_non_seizure_hours if total_non_seizure_hours > 0 else 0
else:
    sensitivity = specificity = fpr_per_hour = None

fpr_vals, tpr_vals, _ = roc_curve(y_true, y_scores)
roc_auc = auc(fpr_vals, tpr_vals)

print("\n=== Final Model Performance on Test Set ===")
print(f"Accuracy:    {final_accuracy:.4f}")
print(f"Precision:   {final_precision:.4f}")
print(f"Recall:      {final_recall:.4f}")
print(f"F1 Score:    {final_f1:.4f}")
print(f"Sensitivity: {sensitivity:.4f}")
print(f"Specificity: {specificity:.4f}")
print(f"AUC-ROC:     {roc_auc:.4f}")
print(f"FPR/h:       {fpr_per_hour:.4f}")
print("\nConfusion Matrix (Test):")
print(final_cm)

plt.figure(figsize=(8, 6))
plt.plot(fpr_vals, tpr_vals, color='blue', lw=2, label=f'ROC Curve (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='gray', linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc='lower right')
plt.show()

plt.figure(figsize=(8, 6))
sns.heatmap(final_cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix (Test)')
plt.show()